# Production RAG Pipeline — Customer Support Domain (techqa / emanual / delucionqa)

This notebook runs the **final, chosen production configuration** end-to-end across all three
customer-support subsets of `galileo-ai/ragbench`, and reports its TRACe scores. It is *not* an
ablation sweep — that work is in `rag-experiments/{dataset}-openrouter-experiment/` — this notebook
exists to demonstrate and validate the single configuration that came out of that sweep as the winner.

## Why this configuration

After a full ablation sweep (up to 11 variants per dataset: baseline, embedder swap, chunking swap,
hybrid dense+sparse fusion with and without reranking, HyDE, step-back, combined-lever configs, and an
externally-inspired wide-retrieval variant) and a rigorous n=100 head-to-head validation on the two
strongest contenders, the plain **dense-only retrieval baseline** won:

- **Chunking**: fixed-word, sized per dataset (128/200/160 words for emanual/techqa/delucionqa)
- **Embedding**: `sentence-transformers/all-MiniLM-L6-v2` (384d) — beat the stronger `BAAI/bge-base-en-v1.5`
  once sample size went beyond n=20
- **Retrieval**: dense-only, top_k=5 — no hybrid fusion, no reranking, no query transform
- **Generation/Judge**: a large capable model (`meta-llama/llama-3.3-70b-instruct` here; `llama-3.3-70b-versatile`
  on Groq) — confirmed to matter far more than any retrieval-side lever, both in our own techqa run and
  independently in a different team's capstone pipeline using a smaller 8B generator

Every attempt to beat this with something fancier either underperformed in the original sweep or lost
the larger-sample validation. See `production/configs/*.yaml` for the per-dataset rationale, and
`production/run_pipeline.py` for a standalone reusable runner (not tied to RAGBench/notebooks).

**Known open limitation, not fixed by this or any config tried**: techqa's real (non-refusal) adherence
stays near-zero regardless of retrieval strategy — a generation-prompt/domain issue, not a retrieval
problem, and worth separate follow-up work.

## 1. Setup & Dependencies

In [1]:
get_ipython().system('pip3 install datasets faiss-cpu sentence-transformers torch groq openai python-dotenv nltk pandas rank_bm25 -q')


[notice] A new release of pip is available: 26.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


## 2. Imports & Project Root

In [2]:
import sys
import os
from pathlib import Path
from dotenv import load_dotenv

# --- Point this at wherever THIS repo (rag_cust_support) lives. ---
# On Colab this is typically under your mounted Drive. Adjust if different.
PROJECT_ROOT = Path('/content/drive/MyDrive/Capstone/rag_cust_support')
if not PROJECT_ROOT.exists():
    # Fallback: running locally from the notebooks/ folder.
    PROJECT_ROOT = Path.cwd().parent if 'notebooks' in str(Path.cwd()) else Path.cwd()

os.chdir(PROJECT_ROOT)
project_root = PROJECT_ROOT
# Make THIS repo win on sys.path (avoids importing a stale rag-foundry copy).
sys.path = [p for p in sys.path if 'rag-foundry' not in p]
if str(project_root) in sys.path:
    sys.path.remove(str(project_root))
sys.path.insert(0, str(project_root))

from experiment.experiment_config import ExperimentConfig
from experiment.experiment_runner import ExperimentRunner
import experiment.experiment_runner as _er, core.registry as _reg

load_dotenv(override=True)
print('Current directory:', Path.cwd())
print('experiment_runner loaded from:', _er.__file__)
assert 'rag-foundry' not in _er.__file__, 'Still importing the old rag-foundry code! Restart runtime.'
print('HuggingFace token loaded:', bool(os.getenv('HF_TOKEN')))
print('Groq API key loaded:', bool(os.getenv('GROQ_API_KEY')))
print('OpenRouter API key loaded:', bool(os.getenv('OPENROUTER_API_KEY')))

/Users/bhupendra.bhoi/pandas_env/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Current directory: /Users/bhupendra.bhoi/aiml/Capstone Project/rag_cust_support
experiment_runner loaded from: /Users/bhupendra.bhoi/aiml/Capstone Project/rag_cust_support/experiment/experiment_runner.py
HuggingFace token loaded: True
Groq API key loaded: True
OpenRouter API key loaded: True


## 3. Run the production config for each dataset

Each dataset has its own isolated experiment dir (`rag-experiments/{dataset}-production/`) containing
**only** the one chosen production config — not the full ablation sweep — driven by
`experiment_configs/{dataset}_production_experiment.yaml`. Defaults to `end_index=20` per dataset for a
quick validation pass; raise it (up to each dataset's full test-split size) for a fuller production
validation once you're ready — the run is checkpoint-resumable, so raising it later won't re-cost
anything already completed.

In [3]:
DATASETS = ['techqa', 'emanual', 'delucionqa']

runners = {}
reports = {}

for dataset in DATASETS:
    print(f'\n{"="*80}\n{dataset.upper()} — production config\n{"="*80}')

    experiment_config = ExperimentConfig.load(project_root / f'experiment_configs/{dataset}_production_experiment.yaml')
    runner = ExperimentRunner(experiment_config)
    runners[dataset] = runner

    documents, raw_data = runner.load_data()
    print(f'Loaded {len(raw_data)} raw rows -> {len(documents)} parsed documents')

    configs = runner.load_configs()
    assert len(configs) == 1, f'Expected exactly 1 production config for {dataset}, found {len(configs)}'
    print(f'Running production config: {configs[0].name}')

    runs = runner.run(documents, raw_data)
    runs = runner.evaluate_runs(runs)
    reports[dataset] = runner.generate_reports(runs)


TECHQA — production config
Loading HuggingFace dataset: galileo-ai/ragbench/techqa (test)...
Loaded 314 samples
Loaded 314 raw rows -> 769 parsed documents
Running production config: techqa_production
OpenRouter provider initialized with 5 key(s) (rotation needs 2+ keys; set OPENROUTER_API_KEY comma-separated).
Loading embedding model: sentence-transformers/all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8082.87it/s]


Progress: 0/20 (0.0%) | QPS: 0.00 | ETA: Unknown | Elapsed: 0sUsing key #0: ****e916Using key #0: ****e916

HTTP 402 (retryable) on ****e916
Error code: 402 - {'error': {'message': 'Insufficient credits. This account never purchased credits. Make sure your key is on the correct account or org, and if so, purchase more at https://openrouter.ai/settings/credits', 'code': 402}}
{'provider': 'openrouter', 'current_index': 0, 'keys': [{'key_suffix': 'e916', 'available': False, 'cooldown_until': datetime.datetime(2026, 7, 26, 17, 33, 8, 463577), 'requests': 2, 'successes': 0, 'failures': 1, '429s': 1}, {'key_suffix': 'b076', 'available': True, 'cooldown_until': None, 'requests': 0, 'successes': 0, 'failures': 0, '429s': 0}, {'key_suffix': 'fbdb', 'available': True, 'cooldown_until': None, 'requests': 0, 'successes': 0, 'failures': 0, '429s': 0}, {'key_suffix': '2462', 'available': True, 'cooldown_until': None, 'requests': 0, 'successes': 0, 'failures': 0, '429s': 0}, {'key_suffix': 'f2e4', '

2026-07-26 17:32:10,093 ERROR rag.pipeline.rag_pipeline: Query failed: No available OpenRouter API keys (all rate-limited/cooling down).
2026-07-26 17:32:10,116 ERROR rag.pipeline.rag_pipeline: Query failed: No available OpenRouter API keys (all rate-limited/cooling down).
2026-07-26 17:32:10,184 ERROR rag.pipeline.rag_pipeline: Query failed: All OpenRouter API keys are currently rate limited.
2026-07-26 17:32:10,185 ERROR rag.pipeline.rag_pipeline: Query failed: All OpenRouter API keys are currently rate limited.
2026-07-26 17:32:10,219 ERROR rag.pipeline.rag_pipeline: Query failed: All OpenRouter API keys are currently rate limited.
2026-07-26 17:32:10,220 ERROR rag.pipeline.rag_pipeline: Query failed: All OpenRouter API keys are currently rate limited.


HTTP 402 (retryable) on ****f2e4
Error code: 402 - {'error': {'message': 'Insufficient credits. This account never purchased credits. Make sure your key is on the correct account or org, and if so, purchase more at https://openrouter.ai/settings/credits', 'code': 402}}
{'provider': 'openrouter', 'current_index': 4, 'keys': [{'key_suffix': 'e916', 'available': False, 'cooldown_until': datetime.datetime(2026, 7, 26, 17, 33, 8, 473579), 'requests': 2, 'successes': 0, 'failures': 2, '429s': 2}, {'key_suffix': 'b076', 'available': False, 'cooldown_until': datetime.datetime(2026, 7, 26, 17, 33, 8, 923438), 'requests': 2, 'successes': 0, 'failures': 2, '429s': 2}, {'key_suffix': 'fbdb', 'available': False, 'cooldown_until': datetime.datetime(2026, 7, 26, 17, 33, 9, 281529), 'requests': 2, 'successes': 0, 'failures': 2, '429s': 2}, {'key_suffix': '2462', 'available': False, 'cooldown_until': datetime.datetime(2026, 7, 26, 17, 33, 9, 727568), 'requests': 2, 'successes': 0, 'failures': 2, '429s'

...

Progress: 20/20 (100.0%) | QPS: 19.88 | ETA: 0s | Elapsed: 1s
[emanual_production] Evaluation complete → rag-experiments/emanual-production/temp/emanual_production.jsonl

DELUCIONQA — production config
Loading HuggingFace dataset: galileo-ai/ragbench/delucionqa (test)...
Loaded 184 samples
Loaded 184 raw rows -> 235 parsed documents
Running production config: delucionqa_production
Progress: 0/20 (0.0%) | QPS: 0.00 | ETA: Unknown | Elapsed: 0s

2026-07-26 17:32:19,203 ERROR rag.pipeline.rag_pipeline: Query failed: All OpenRouter API keys are currently rate limited.
2026-07-26 17:32:19,203 ERROR rag.pipeline.rag_pipeline: Query failed: All OpenRouter API keys are currently rate limited.
2026-07-26 17:32:19,375 ERROR rag.pipeline.rag_pipeline: Query failed: All OpenRouter API keys are currently rate limited.
2026-07-26 17:32:19,376 ERROR rag.pipeline.rag_pipeline: Query failed: All OpenRouter API keys are currently rate limited.
2026-07-26 17:32:19,503 ERROR rag.pipeline.rag_pipeline: Query failed: All OpenRouter API keys are currently rate limited.


Progress: 5/20 (25.0%) | QPS: 4.98 | ETA: 3s | Elapsed: 1s

2026-07-26 17:32:20,367 ERROR rag.pipeline.rag_pipeline: Query failed: All OpenRouter API keys are currently rate limited.
2026-07-26 17:32:20,367 ERROR rag.pipeline.rag_pipeline: Query failed: All OpenRouter API keys are currently rate limited.
2026-07-26 17:32:20,385 ERROR rag.pipeline.rag_pipeline: Query failed: All OpenRouter API keys are currently rate limited.
2026-07-26 17:32:20,385 ERROR rag.pipeline.rag_pipeline: Query failed: All OpenRouter API keys are currently rate limited.
2026-07-26 17:32:20,550 ERROR rag.pipeline.rag_pipeline: Query failed: All OpenRouter API keys are currently rate limited.
2026-07-26 17:32:20,550 ERROR rag.pipeline.rag_pipeline: Query failed: All OpenRouter API keys are currently rate limited.
2026-07-26 17:32:20,577 ERROR rag.pipeline.rag_pipeline: Query failed: All OpenRouter API keys are currently rate limited.
2026-07-26 17:32:20,578 ERROR rag.pipeline.rag_pipeline: Query failed: All OpenRouter API keys are currently rate limited.
2026-07-26 17:32

Progress: 20/20 (100.0%) | QPS: 9.95 | ETA: 0s | Elapsed: 2s
[delucionqa_production] Evaluation complete → rag-experiments/delucionqa-production/temp/delucionqa_production.jsonl


## 4. Results

In [4]:
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 80)

for dataset in DATASETS:
    print(f'\n{"="*80}\n{dataset.upper()}\n{"="*80}')
    comparison = runners[dataset].compare()
    display(comparison.to_dataframe())


TECHQA


,config_name,relevance_score__mean,relevance_score__mae,utilization_score__mean,utilization_score__mae,completeness_score__mean,completeness_score__mae,adherence_score__mean,adherence_score__mae
0,techqa_production,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



EMANUAL


,config_name,relevance_score__mean,relevance_score__mae,utilization_score__mean,utilization_score__mae,completeness_score__mean,completeness_score__mae,adherence_score__mean,adherence_score__mae
0,emanual_production,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



DELUCIONQA


,config_name,relevance_score__mean,relevance_score__mae,utilization_score__mean,utilization_score__mae,completeness_score__mean,completeness_score__mae,adherence_score__mean,adherence_score__mae
0,delucionqa_production,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 5. Conclusion

This is the configuration recommended for the customer-support domain going forward. It is deliberately
the *simplest* option in the whole sweep — no hybrid fusion, no reranker, no query transform — which won
specifically because every added layer of retrieval sophistication either underperformed in the original
sweep or lost the head-to-head validation against it. For real reuse outside this benchmark/notebook
context (e.g. against your own documents), see `production/run_pipeline.py`.